# Aegis SFT E4B v2 ? merge LoRA from HF Hub

Use this notebook after a Colab reset when the only surviving artifact is the LoRA adapter repo:

- LoRA: `V1rtucious/aegis-sft-e4b-lora-v2`
- Merged target: `V1rtucious/aegis-sft-e4b-merged-v2`

Workflow:
1. Install/load Unsloth.
2. Load the LoRA adapter from HF Hub.
3. Run the mode-contract smoke test against the adapter.
4. Merge to `/content/merged-v2`.
5. Optionally reload/smoke the merged folder.
6. Push merged FP16 to HF Hub.

Use an A100/L4/T4 GPU runtime. A100 is fastest; T4 works but is slower.

## Cell 1 ? Install deps

In [ ]:
# Unsloth merge/runtime deps
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U huggingface_hub accelerate bitsandbytes

import torch, transformers
print(f"torch        : {torch.__version__} cuda={torch.cuda.is_available()}")
print(f"transformers : {transformers.__version__}")
if torch.cuda.is_available():
    print(f"gpu          : {torch.cuda.get_device_name(0)}")
    print(f"bf16 support : {torch.cuda.is_bf16_supported()}")

## Cell 2 ? Secrets + constants

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
login(token=os.environ["HF_TOKEN"])

LORA_REPO = "V1rtucious/aegis-sft-e4b-lora-v2"
MERGED_REPO = "V1rtucious/aegis-sft-e4b-merged-v2"
MERGED_DIR = "/content/merged-v2"
MAX_SEQ_LENGTH = 2048

print("LoRA repo   :", LORA_REPO)
print("Merged repo :", MERGED_REPO)
print("Merged dir  :", MERGED_DIR)

## Cell 3 ? Load LoRA adapter from HF Hub

This loads the adapter repo directly. If this fails, the LoRA repo may not contain the adapter metadata/tokenizer needed by Unsloth.

In [ ]:
import os, torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_REPO,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)
model.eval()


def resolve_text_tokenizer(processor_or_tokenizer):
    if hasattr(processor_or_tokenizer, "encode"):
        return processor_or_tokenizer
    for attr in ("tokenizer", "text_tokenizer"):
        if hasattr(processor_or_tokenizer, attr):
            inner = getattr(processor_or_tokenizer, attr)
            if hasattr(inner, "encode"):
                return inner
    raise AttributeError(type(processor_or_tokenizer).__name__)

text_tokenizer = resolve_text_tokenizer(tokenizer)
print("Loaded adapter model from:", LORA_REPO)
print("Tokenizer:", type(text_tokenizer).__name__)
try:
    print("PEFT config:", model.peft_config)
except Exception as exc:
    print("PEFT config unavailable:", exc)

## Cell 4 ? Get tool definitions

In [ ]:
import json, os
from huggingface_hub import hf_hub_download

TOOL_DEFS_PATH = "tool_defs.json"
if not os.path.exists(TOOL_DEFS_PATH):
    try:
        TOOL_DEFS_PATH = hf_hub_download(
            "V1rtucious/aegis-sft-data",
            "tool_defs.json",
            repo_type="dataset",
            local_dir=".",
        )
    except Exception:
        from google.colab import files
        print("Upload tool_defs.json from tools/tools/tool_defs.json")
        files.upload()
        TOOL_DEFS_PATH = "tool_defs.json"

with open(TOOL_DEFS_PATH) as f:
    TOOL_DEFS_LIST = json.load(f)
print(f"Loaded {len(TOOL_DEFS_LIST)} tool definitions from {TOOL_DEFS_PATH}")

## Cell 5 ? Smoke test adapter

This is first-turn contract smoke only. It verifies the adapter still has the behavior we saw before merging:
- DrugSafe emits native tool calls.
- ConsentReader emits full JSON with no tools.
- HealthPartner prevention emits `get_guideline`.
- HealthPartner symptom emits full JSON with no tools.
- `<|channel>` thought output is banned at generation time.

In [ ]:
import json, re, torch

SMOKE_TESTS = [
    ("drugsafe", "tool_call", "I'm on warfarin. Is it safe to take aspirin for a headache?"),
    ("drugsafe", "tool_call", "My 8-year-old has a fever. Can I give him aspirin?"),
    ("drugsafe", "tool_call", "I take buprenorphine. Can I take pseudoephedrine for a cold?"),
    ("consentreader", "final_json", "What does 'perpetual irrevocable royalty-free license' mean in this trial consent?"),
    ("healthpartner", "tool_call", "I'm a 50yo male - what preventive screenings do I need?"),
    ("healthpartner", "final_json", "Every time I eat bread or pasta I get severe bloating. Do I have celiac disease?"),
]

REQUIRED = {"flags", "citations", "confidence", "defer_to_professional", "explanation"}


def _uses_tools(mode, expected):
    return expected == "tool_call" and mode != "consentreader"


def _system_prompt(mode, expected):
    base = (
        "You are Aegis Health, an offline medical safety assistant running locally on the user's device. "
        "You have NO internet access. Respond with ONLY the required output: a native tool_call when a tool is needed, "
        "or exactly one final JSON object with this schema: "
        "{\"flags\":[{\"severity\":<1-5>,\"description\":\"...\",\"citation\":\"...\"}],"
        "\"citations\":[{\"source\":\"...\",\"text\":\"...\"}],"
        "\"confidence\":<0.0-1.0>,\"defer_to_professional\":<true|false>,\"explanation\":\"...\"}. "
        "If no safety flags apply, use flags: []. "
        "Do NOT emit <|channel>thought, analysis, markdown, or prose outside JSON.\n\n"
    )
    if mode == "drugsafe":
        return base + (
            "Mode: DrugSafe\n"
            "Use tools. Prefer one check_warnings call with the complete drug_list. "
            "Use normalize_drug/decompose_product only when names are unclear."
        )
    if mode == "consentreader":
        return base + (
            "Mode: ConsentReader\n"
            "No tools are available in this mode. Do not emit tool calls. "
            "Return final JSON that explains the consent language plainly."
        )
    if expected == "final_json":
        return base + (
            "Mode: HealthPartner\n"
            "This is an active symptom or diagnosis question. Do not call get_guideline. "
            "Return exactly this JSON shape, filling in concise case-specific strings: "
            "{\"flags\":[{\"severity\":3,\"description\":\"Cannot diagnose from symptoms alone; clinical evaluation/testing is needed.\","
            "\"citation\":\"Aegis safety policy\"}],"
            "\"citations\":[{\"source\":\"Aegis safety policy\",\"text\":\"Do not diagnose active symptoms; recommend clinician evaluation.\"}],"
            "\"confidence\":0.8,\"defer_to_professional\":true,"
            "\"explanation\":\"I cannot diagnose this condition from symptoms alone. Please discuss appropriate testing with a clinician.\"}"
        )
    return base + (
        "Mode: HealthPartner\n"
        "Use get_guideline only for preventive screening/profile questions with age and sex."
    )

BAD_WORDS = [
    text_tokenizer.encode("<|channel>thought", add_special_tokens=False),
    text_tokenizer.encode("<|channel>", add_special_tokens=False),
]
BAD_WORDS = [ids for ids in BAD_WORDS if ids]
print("bad_words_ids:", BAD_WORDS)

@torch.no_grad()
def _gen_one(prompt_str, max_new_tokens=512):
    inputs = text_tokenizer(prompt_str, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=text_tokenizer.eos_token_id,
        bad_words_ids=BAD_WORDS,
    )
    new = out[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(new, skip_special_tokens=False).strip()

NATIVE_TC_RE = re.compile(r"<\|tool_call>\s*call:\s*(\w+)\s*\{(.*?)\}\s*<tool_call\|>", re.DOTALL)
HYBRID_GARBAGE_RE = re.compile(r'"arguments"\s*:\s*\{[^}]*\}\s*[\]\)\}]?\s*,\s*"result"\s*:')


def _parse_native_args(args_str):
    s = args_str.replace('<|"|>', '"')
    s = re.sub(r'(?<!["\w])(\w+)\s*:', r'"\1":', s)
    try:
        return json.loads("{" + s + "}")
    except json.JSONDecodeError:
        return None


def _clean_final_text(text):
    text = re.sub(r"(?:\s*(?:<turn\|>|<eos>))+\s*$", "", text)
    text = re.sub(r"^\s*<\|turn>model\s*\n?", "", text)
    return text.strip()


def diagnose(output):
    tc_matches = list(NATIVE_TC_RE.finditer(output))
    valid_calls, invalid_calls = [], []
    for m in tc_matches:
        parsed = _parse_native_args(m.group(2))
        if parsed is None:
            invalid_calls.append({"name": m.group(1), "raw": m.group(2)})
        else:
            valid_calls.append({"name": m.group(1), "args": parsed})
    cleaned = _clean_final_text(output)
    try:
        parsed_final = json.loads(cleaned)
    except Exception:
        parsed_final = None
    return {
        "n_tool_calls": len(tc_matches),
        "n_valid_native_calls": len(valid_calls),
        "n_invalid_native_calls": len(invalid_calls),
        "tool_names": [c["name"] for c in valid_calls],
        "has_partial_tool_response_start": "<|tool_response>" in output,
        "has_hybrid_garbage": bool(HYBRID_GARBAGE_RE.search(output)),
        "has_final_json": isinstance(parsed_final, dict) and REQUIRED.issubset(parsed_final),
        "has_thought_channel": "<|channel>" in output,
        "raw_length": len(output),
    }

failures, results = [], []
print("=" * 72)
print(" Adapter smoke test - mode-specific first-turn contract")
print("=" * 72)
for i, (mode, expected, user_msg) in enumerate(SMOKE_TESTS, 1):
    messages = [
        {"role": "system", "content": _system_prompt(mode, expected)},
        {"role": "user", "content": user_msg},
    ]
    kwargs = {"tokenize": False, "add_generation_prompt": True}
    if _uses_tools(mode, expected):
        kwargs["tools"] = TOOL_DEFS_LIST
    formatted = tokenizer.apply_chat_template(messages, **kwargs)
    output = _gen_one(formatted)
    diag = diagnose(output)
    results.append(diag)

    if expected == "tool_call":
        ok = diag["n_valid_native_calls"] > 0 and diag["n_invalid_native_calls"] == 0 and not diag["has_hybrid_garbage"] and not diag["has_thought_channel"]
    else:
        ok = diag["n_tool_calls"] == 0 and diag["has_final_json"] and not diag["has_hybrid_garbage"] and not diag["has_thought_channel"]
    if not ok:
        failures.append((i, mode, expected, diag, output[:400]))

    print(f"\n[{i}/{len(SMOKE_TESTS)}] mode={mode} expected={expected} prompt={user_msg[:70]!r}")
    print("  diagnostics:", diag)
    print("  output first 500 chars:")
    for line in output[:500].split("\n"):
        print("    |", line)

print("\n" + "=" * 72)
print(" Verdict")
print("=" * 72)
print(f"  passed: {len(SMOKE_TESTS) - len(failures)}/{len(SMOKE_TESTS)}")
if failures:
    for idx, mode, expected, diag, preview in failures:
        print(f"  FAIL case {idx} mode={mode} expected={expected}: {diag}")
        print("  preview:", repr(preview))
    raise RuntimeError("Adapter smoke failed; do not merge/push yet.")
print("PASS: adapter first-turn contract looks sane.")


## Cell 6 ? Merge LoRA to FP16/BF16 checkpoint

This writes the merged model to `/content/merged-v2`. It may take several minutes and uses substantial disk.

In [ ]:
import os
os.makedirs(MERGED_DIR, exist_ok=True)

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)
!du -sh /content/merged-v2
!ls -lh /content/merged-v2 | head -30

## Cell 7 ? Optional: reload merged folder and run the same smoke

This catches a bad merge before uploading 15 GB. If memory is tight, restart the runtime and skip this cell; eval will load the pushed merged model later.

In [ ]:
# Optional but recommended on A100/L4. On T4, this may be slow or memory tight.
import gc, torch

try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
except Exception:
    pass

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)
model.eval()
text_tokenizer = resolve_text_tokenizer(tokenizer)
print("Reloaded merged model from", MERGED_DIR)

# Rerun Cell 5 after this cell to smoke the merged model.


## Cell 8 ? Push merged FP16 to HF Hub

In [ ]:
from huggingface_hub import HfApi, create_repo

create_repo(MERGED_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
HfApi().upload_folder(
    folder_path=MERGED_DIR,
    repo_id=MERGED_REPO,
    repo_type="model",
    token=os.environ["HF_TOKEN"],
)
print(f"Merged FP16 pushed: https://huggingface.co/{MERGED_REPO}")

## Cell 9 ? Next step

Run `eval_heldout.ipynb` with the newest `aegis-eval-kit.tar.gz`, and make sure the SFT cells point to:

```python
repo_id="V1rtucious/aegis-sft-e4b-merged-v2"
local_dir="/content/sft-merged-v2"
tag="sft-v2"
```

Do not run LitertLM export until held-out eval is acceptable.